In [ ]:
#Võ Hoàn Lạc

In [ ]:
#Tiền xử lý dữ liệu
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
#  Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

DATA_DIR = "/content/drive/MyDrive/datadoan-20251228T134449Z-1-001/datadoan/plant-health" # Corrected path to the class subfolders

dataset = datasets.ImageFolder(DATA_DIR, transform=transform)
class_names = dataset.classes
print("Classes:", class_names)



Mounted at /content/drive
Classes: ['healthy', 'unhealthy']


In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
data = []
labels = []

for label, cls in enumerate(class_names):
    folder = os.path.join(DATA_DIR, cls)
    if not os.path.isdir(folder):
        print(f"Warning: Directory not found: {folder}")
        continue
    for img_name in os.listdir(folder):
        img_path = os.path.join(folder, img_name)
        img = cv2.imread(img_path)
        if img is None:
            print(f"Warning: Could not read image {img_path}")
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        data.append(img)
        labels.append(label)

X = np.array(data, dtype="float32")
y = np.array(labels)


In [ ]:
X = X / 255.0


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=30,
    zoom_range=0.2,
    horizontal_flip=True
)

datagen.fit(X_train)


In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.models import Model

base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

for layer in base_model.layers:
    layer.trainable = False


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [ ]:
x = base_model.output
x = Flatten()(x)
x = Dropout(0.5)(x)
x = Dense(128, activation="relu")(x)
output = Dense(1, activation="sigmoid")(x)

model = Model(inputs=base_model.input, outputs=output)


In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
history = model.fit(
    datagen.flow(X_train, y_train, batch_size=32),
    validation_data=(X_val, y_val),
    epochs=20
)


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 149s 9s/step - accuracy: 0.5422 - loss: 3.6615 - val_accuracy: 0.6038 - val_loss: 0.8332
Epoch 2/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 126s 8s/step - accuracy: 0.6212 - loss: 0.8123 - val_accuracy: 0.6981 - val_loss: 0.6099
Epoch 3/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 126s 8s/step - accuracy: 0.6083 - loss: 0.7647 - val_accuracy: 0.5660 - val_loss: 0.8580
Epoch 4/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 125s 8s/step - accuracy: 0.5910 - loss: 0.9386 - val_accuracy: 0.6981 - val_loss: 0.6775
Epoch 5/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 142s 8s/step - accuracy: 0.6064 - loss: 0.7403 - val_accuracy: 0.7170 - val_loss: 0.6192
Epoch 6/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 157s 10s/step - accuracy: 0.6320 - loss: 0.7909 - val_accuracy: 0.7453 - val_loss: 0.5987
Epoch 7/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7s/step - accuracy: 0.6337 - loss: 0.7582